In [30]:
from pathlib import Path
import pandas as pd
import os
import shutil
import hashlib

In [31]:
os.getcwd()

'/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification'

In [4]:
os.chdir("..")

In [5]:
os.getcwd()

'/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification'

In [5]:
ROOT = Path.cwd()

IN_ROOT = ROOT / "data"/ "raw" / "raw_reworked"
OUT_ROOT = ROOT / "data" / "Consolidated"

STAGE1_ROOT = OUT_ROOT / "Stage-1"
STAGE2_ROOT = OUT_ROOT / "Stage-2"


IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# Mapping defined earlier
MAPPING = {
    "cardboard":  ("Non_Organic", "Cardboard"),
    "white_glass":      ("Non_Organic", "White_Glass"),
    "brown_glass":      ("Non_Organic", "Brown_Glass"),
    "green_glass":      ("Non_Organic", "Green_Glass"),
    "metal":      ("Non_Organic", "Metal"),
    "paper":      ("Non_Organic", "Paper"),
    "plastic":    ("Non_Organic", "Plastic"),
    "textile":    ("Non_Organic", "Textile"),
    "battery":    ("Non_Organic", "Battery"),
    "shoes":    ("Non_Organic", "Shoes"),
    "e_waste":  ("Non_Organic", "E_waste"),
    "medical_waste":  ("Non_Organic", "Medical_waste"),
    "misc_trash":      ("Non_Organic", "Misc_Trash"),
    "food_organics":    ("Organic", None),
    "vegetation":    ("Organic", None),
    "organics":    ("Organic", None),
    "non_organics":    ("Non_Organic", None),
}

def unique_id(path: Path) -> str:
    h = hashlib.sha1(str(path).encode()).hexdigest()
    return h[:8]

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

for SRC_ROOT in IN_ROOT.iterdir():
    if not SRC_ROOT.is_dir():
        continue

    SRC_name = SRC_ROOT.name

    print("Begin processin dataset :", SRC_name)

    for class_dir in SRC_ROOT.iterdir():
        if not class_dir.is_dir():
            continue
    
        class_name = class_dir.name
        print("Class :", class_name)
    
        if class_name not in MAPPING:
            print(f"Skipping unmapped folder: {class_dir.name}")
            continue
    
        stage1, stage2 = MAPPING[class_name]
    
        # Create destination folders
        stage1_dir = STAGE1_ROOT / stage1
    
        if stage2:
            stage2_dir = STAGE2_ROOT / stage2

        safe_mkdir(stage1_dir)
        safe_mkdir(stage2_dir)
    
        for img in class_dir.iterdir():
            if img.suffix.lower() not in IMG_EXTS:
                continue
    
            uid = unique_id(img)
            ext = img.suffix.lower()
    
            # Filename
            if stage2:
                new_name = f"{SRC_name}_{stage1}_{stage2}_{uid}{ext}"
            else:
                new_name = f"{SRC_name}_{stage1}_{uid}{ext}"
    
            # Copy to Stage-1
            shutil.copy2(img, stage1_dir / new_name)
    
            # Copy to Stage-2 (if applicable)
            if stage2:
                shutil.copy2(img, stage2_dir / new_name)

    print("Done processin dataset :", SRC_name)


Begin processin dataset : realwaste-main
Class : paper
Class : misc_trash
Class : textile
Class : food_organics
Class : metal
Class : cardboard
Class : green_glass
Class : brown_glass
Class : vegetation
Class : plastic
Class : shoes
Class : white_glass
Done processin dataset : realwaste-main
Begin processin dataset : MendleyGCdataset
Class : organics
Class : non_organics
Done processin dataset : MendleyGCdataset
Begin processin dataset : garbage_classification
Class : organics
Class : paper
Class : misc_trash
Class : textile
Class : metal
Class : cardboard
Class : green_glass
Class : brown_glass
Class : battery
Class : plastic
Class : shoes
Class : white_glass
Done processin dataset : garbage_classification
Begin processin dataset : Trashnet
Class : organics
Class : paper
Class : misc_trash
Class : metal
Class : cardboard
Class : green_glass
Class : brown_glass
Class : plastic
Class : white_glass
Done processin dataset : Trashnet
Begin processin dataset : Warp-main
Class : metal
Class 

In [6]:
BASE_DIR = Path.cwd()
STAGE1_DIR = BASE_DIR / "data" / "consolidated" / "Stage-1"
STAGE2_DIR = BASE_DIR / "data" / "consolidated" / "Stage-2"
META_DIR = BASE_DIR / "data" / "metadata"

# Mapping defined earlier
TYPE_MAPPING = {
    "garbage":  ("clean"),
    "realwaste-main":      ("real_world"),
    "MendleyGCdataset":      ("clean"),
    "TACO":      ("real_world"),
    "Trashnet":      ("clean"),
    "Tricascade":      ("clean"),
    "Warp-main":    ("real_world"),
}

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def create_metadata(stage1_dir: Path, stage2_dir: Path):
    
    data = []

    for class_folder in sorted(stage1_dir.iterdir()):
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name

        for img_path in class_folder.rglob("*"):
            if img_path.suffix.lower() in VALID_EXTS:

                filename = img_path.name
                source = filename.split("_")[0] if "_" in filename else "unknown"

                type_image= TYPE_MAPPING[source]

                if class_name == 'Non_Organic' and not filename.startswith("Mendley") :
                    continue;

                split_key = class_name + '_' + type_image 
            
                data.append({
                "filepath": str(img_path.resolve()),
                "stage1_label": class_name,
                "stage2_label": class_name,
                "source": source,
                "type":  type_image,
                "split_key" : split_key
                
            })

    print(f"\nCreated metadata for {class_folder.name}")
    
    for class_folder in sorted(stage2_dir.iterdir()):
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name

        for img_path in class_folder.rglob("*"):
            if img_path.suffix.lower() in VALID_EXTS:

                filename = img_path.name
                source = filename.split("_")[0] if "_" in filename else "unknown"

                type_image= TYPE_MAPPING[source]

                split_key = class_name + '_' + type_image
            
                data.append({
                "filepath": str(img_path.resolve()),
                "stage1_label": "Non_Organic",
                "stage2_label": class_name,
                "source": source,
                "type":  type_image,
                "split_key" : split_key
                })
                
        print(f"\nAppended metadata for {class_folder.name}")    

    df = pd.DataFrame(data)

    
    print(f"Total samples: {len(df)}")
    print("Class distribution Stage 1:")
    print(df["stage1_label"].value_counts())
    print("Class distribution Stage 2:")
    print(df["stage2_label"].value_counts())
    
    return df

metadata_df = create_metadata(STAGE1_DIR, STAGE2_DIR) 

metadata_df.to_csv(META_DIR / "consolidated_metatdata-dup-clean.csv", index=False)

print("\n consolidated metadata files created")


Created metadata for Organic_Augmented

Appended metadata for Battery

Appended metadata for Brown_Glass

Appended metadata for Cardboard

Appended metadata for E_waste

Appended metadata for Green_Glass

Appended metadata for Medical_waste

Appended metadata for Metal

Appended metadata for Misc_Trash

Appended metadata for Paper

Appended metadata for Plastic

Appended metadata for Shoes

Appended metadata for Textile

Appended metadata for White_Glass
Total samples: 63386
Class distribution Stage 1:
stage1_label
Non_Organic    45074
Organic        18312
Name: count, dtype: int64
Class distribution Stage 2:
stage2_label
Organic          18312
Non_Organic      11055
Plastic          10512
Textile           5535
Metal             2471
Cardboard         2306
E_waste           2278
Shoes             2062
Medical_waste     1699
Paper             1629
Misc_Trash        1494
White_Glass       1389
Battery            941
Green_Glass        864
Brown_Glass        839
Name: count, dtype: int6

In [17]:
BASE_DIR = Path.cwd()
STAGE1_DIR = BASE_DIR / "data" / "consolidated" / "Stage-1"
STAGE2_DIR = BASE_DIR / "data" / "consolidated" / "Stage-2"
META_DIR = BASE_DIR / "data" / "metadata"

# Mapping defined earlier
TYPE_MAPPING = {
    "garbage":  ("clean"),
    "realwaste-main":      ("real_world"),
    "MendleyGCdataset":      ("clean"),
    "TACO":      ("real_world"),
    "Trashnet":      ("clean"),
    "Tricascade":      ("clean"),
    "Warp-main":    ("real_world"),
}

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def create_metadata(stage1_dir: Path, stage2_dir: Path):
    
    data_stg1 = []
    data_stg2 = []

    for class_folder in sorted(stage1_dir.iterdir()):
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name
        print(f"\Creating metadata for Stage 1 : {class_folder.name}")
        
        for img_path in class_folder.rglob("*"):
            if img_path.suffix.lower() in VALID_EXTS:

                filename = img_path.name
                source = filename.split("_")[0] if "_" in filename else "unknown"

                type_image= TYPE_MAPPING[source]

                #if class_name == 'Non_Organic' and not filename.startswith("Mendley") :
                #    continue;

                split_key = class_name + '_' + type_image 
            
                data_stg1.append({
                "filepath": str(img_path.resolve()),
                "stage1_label": class_name,
                "stage2_label": "",
                "source": source,
                "type":  type_image,
                "split_key" : split_key
                
            })

    
    
    for class_folder in sorted(stage2_dir.iterdir()):
        if not class_folder.is_dir():
            continue

        class_name = class_folder.name
        print(f"\Creating metadata for Stage 2 : {class_folder.name}")
        for img_path in class_folder.rglob("*"):
            if img_path.suffix.lower() in VALID_EXTS:

                filename = img_path.name
                source = filename.split("_")[0] if "_" in filename else "unknown"

                type_image= TYPE_MAPPING[source]

                split_key = class_name + '_' + type_image
            
                data_stg2.append({
                "filepath": str(img_path.resolve()),
                "stage1_label": "Non_Organic",
                "stage2_label": class_name,
                "source": source,
                "type":  type_image,
                "split_key" : split_key
                })
                
        #print(f"\nAppended metadata for {class_folder.name}")    

    df_stg1 = pd.DataFrame(data_stg1)
    df_stg2 = pd.DataFrame(data_stg2)
    
    print(f"Total samples stage 1: {len(df_stg1)}")
    print(f"Total samples stage 2: {len(df_stg2)}")
    print("Class distribution Stage 1:")
    print(df_stg1["stage1_label"].value_counts())
    print("Class distribution Stage 2:")
    print(df_stg2["stage2_label"].value_counts())
    
    return df_stg1, df_stg2

metadata_stg1_df,metadata_stg2_df  = create_metadata(STAGE1_DIR, STAGE2_DIR) 

metadata_stg1_df.to_csv(META_DIR / "consolidated_metatdata-stg1.csv", index=False)
metadata_stg2_df.to_csv(META_DIR / "consolidated_metatdata-stg2.csv", index=False)

print("\n consolidated metadata files created")

\Creating metadata for Stage 1 : Non_Organic
\Creating metadata for Stage 1 : Organic
\Creating metadata for Stage 2 : Battery
\Creating metadata for Stage 2 : Brown_Glass
\Creating metadata for Stage 2 : Cardboard
\Creating metadata for Stage 2 : E_waste
\Creating metadata for Stage 2 : Green_Glass
\Creating metadata for Stage 2 : Medical_waste
\Creating metadata for Stage 2 : Metal
\Creating metadata for Stage 2 : Misc_Trash
\Creating metadata for Stage 2 : Paper
\Creating metadata for Stage 2 : Plastic
\Creating metadata for Stage 2 : Shoes
\Creating metadata for Stage 2 : Textile
\Creating metadata for Stage 2 : White_Glass
Total samples stage 1: 63271
Total samples stage 2: 33989
Class distribution Stage 1:
stage1_label
Non_Organic    45166
Organic        18105
Name: count, dtype: int64
Class distribution Stage 2:
stage2_label
Plastic          10519
Textile           5524
Metal             2466
Cardboard         2300
E_waste           2277
Shoes             2068
Medical_waste     

In [18]:
metadata_stg1_df

,filepath,stage1_label,stage2_label,source,type,split_key
0,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,,garbage,clean,Non_Organic_clean
1,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,,Warp-main,real_world,Non_Organic_real_world
2,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,,Warp-main,real_world,Non_Organic_real_world
3,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,,Warp-main,real_world,Non_Organic_real_world
4,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,,Tricascade,clean,Non_Organic_clean
...,...,...,...,...,...,...
63266,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,,MendleyGCdataset,clean,Organic_clean
63267,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,,Tricascade,clean,Organic_clean
63268,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,,MendleyGCdataset,clean,Organic_clean
63269,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,,MendleyGCdataset,clean,Organic_clean


In [19]:
metadata_stg2_df

,filepath,stage1_label,stage2_label,source,type,split_key
0,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Battery,garbage,clean,Battery_clean
1,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Battery,garbage,clean,Battery_clean
2,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Battery,garbage,clean,Battery_clean
3,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Battery,garbage,clean,Battery_clean
4,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Battery,garbage,clean,Battery_clean
...,...,...,...,...,...,...
33984,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,Warp-main,real_world,White_Glass_real_world
33985,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean
33986,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean
33987,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean


In [ ]:
metadata_df['stage1_label'].value_counts()

In [7]:
metadata_df['stage1_label'].value_counts()

stage1_label
Non_Organic    45074
Organic        18312
Name: count, dtype: int64

In [8]:
metadata_df

,filepath,stage1_label,stage2_label,source,type,split_key
0,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
1,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
2,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
3,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
4,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
...,...,...,...,...,...,...
63381,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,Warp-main,real_world,White_Glass_real_world
63382,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean
63383,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean
63384,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean


In [9]:
metadata_df.stage1_label.value_counts()


stage1_label
Non_Organic    47681
Organic        19925
Name: count, dtype: int64

In [9]:
metadata_df.stage2_label.value_counts()

stage2_label
Organic          18312
Non_Organic      11055
Plastic          10512
Textile           5535
Metal             2471
Cardboard         2306
E_waste           2278
Shoes             2062
Medical_waste     1699
Paper             1629
Misc_Trash        1494
White_Glass       1389
Battery            941
Green_Glass        864
Brown_Glass        839
Name: count, dtype: int64

In [11]:
metadata_df.duplicated("filepath").sum()

0

In [19]:
metadata_df["exists"] = metadata_df.filepath.apply(lambda x: Path(x).exists())
metadata_df.exists.value_counts()

exists
True    67606
Name: count, dtype: int64

#### Startified split

In [20]:
BASE_DIR = Path.cwd()
META_DIR = BASE_DIR / "data" / "metadata"
file1=META_DIR/"consolidated_metatdata-stg1.csv"

In [21]:
metadata_df = pd.read_csv(file1)

In [22]:
metadata_df

,filepath,stage1_label,stage2_label,source,type,split_key
0,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,NaN,garbage,clean,Non_Organic_clean
1,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,NaN,Warp-main,real_world,Non_Organic_real_world
2,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,NaN,Warp-main,real_world,Non_Organic_real_world
3,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,NaN,Warp-main,real_world,Non_Organic_real_world
4,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,NaN,Tricascade,clean,Non_Organic_clean
...,...,...,...,...,...,...
63266,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,NaN,MendleyGCdataset,clean,Organic_clean
63267,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,NaN,Tricascade,clean,Organic_clean
63268,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,NaN,MendleyGCdataset,clean,Organic_clean
63269,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,NaN,MendleyGCdataset,clean,Organic_clean


In [24]:
from sklearn.model_selection import train_test_split


train_df, temp_df = train_test_split(
    metadata_df,
    test_size=0.30,          # 70 train / 30 temp
    stratify=metadata_df["stage1_label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,          # 15 / 15
    stratify=temp_df["stage1_label"],
    random_state=42
)

In [25]:
train_df.shape

(44289, 6)

In [26]:
val_df.shape

(9491, 6)

In [27]:
test_df.shape

(9491, 6)

In [28]:
train_df['stage1_label'].value_counts()

stage1_label
Non_Organic    31616
Organic        12673
Name: count, dtype: int64

In [29]:
val_df['stage1_label'].value_counts()

stage1_label
Non_Organic    6775
Organic        2716
Name: count, dtype: int64

In [23]:
train_df['stage2_label'].value_counts()

stage2_label
Organic          13947
Non_Organic       7778
Plastic           7724
Textile           3876
Metal             2018
Cardboard         1899
E_waste           1681
Paper             1558
Shoes             1450
Medical_waste     1198
White_Glass       1160
Misc_Trash        1049
Green_Glass        675
Battery            662
Brown_Glass        649
Name: count, dtype: int64

In [25]:
train_df['split_key'].value_counts()

split_key
Organic_clean             13344
Non_Organic_clean          7778
Plastic_real_world         6770
Textile_clean              3693
E_waste_clean              1681
Shoes_clean                1403
Metal_real_world           1212
Medical_waste_clean        1198
Paper_clean                1182
Cardboard_real_world        991
Plastic_clean               954
Cardboard_clean             908
Metal_clean                 806
White_Glass_clean           735
Battery_clean               662
Organic_real_world          603
Misc_Trash_clean            546
Green_Glass_clean           521
Misc_Trash_real_world       503
Brown_Glass_clean           487
White_Glass_real_world      425
Paper_real_world            376
Textile_real_world          183
Brown_Glass_real_world      162
Green_Glass_real_world      154
Shoes_real_world             47
Name: count, dtype: int64

In [26]:
metadata_df['split_key'].value_counts()

split_key
Organic_clean             19070
Non_Organic_clean         11111
Plastic_real_world         9674
Textile_clean              5290
E_waste_clean              2402
Shoes_clean                2010
Medical_waste_clean        1712
Metal_real_world           1704
Paper_clean                1674
Cardboard_real_world       1419
Plastic_clean              1360
Cardboard_clean            1294
Metal_clean                1179
White_Glass_clean          1054
Battery_clean               945
Organic_real_world          855
Misc_Trash_clean            786
Green_Glass_clean           744
Misc_Trash_real_world       713
Brown_Glass_clean           701
White_Glass_real_world      603
Paper_real_world            551
Textile_real_world          247
Brown_Glass_real_world      226
Green_Glass_real_world      220
Shoes_real_world             62
Name: count, dtype: int64

#### Roughly the splits at a class split key level is same between metadata and training dataset

In [27]:
train_df

,filepath,stage1_label,stage2_label,source,type,split_key
66187,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,realwaste-main,real_world,White_Glass_real_world
40152,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Medical_waste,Tricascade,clean,Medical_waste_clean
58557,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Shoes,realwaste-main,real_world,Shoes_real_world
9533,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
26595,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,Organic,Tricascade,clean,Organic_clean
...,...,...,...,...,...,...
20575,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,Organic,MendleyGCdataset,clean,Organic_clean
2625,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
2228,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean
27562,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Organic,Organic,MendleyGCdataset,clean,Organic_clean


In [29]:
metadata_df.loc[metadata_df["filepath"].isin(train_df["filepath"]), "split"] = "train"
metadata_df.loc[metadata_df["filepath"].isin(test_df["filepath"]), "split"] = "test"
metadata_df.loc[metadata_df["filepath"].isin(val_df["filepath"]), "split"] = "val"

In [30]:
metadata_df

,filepath,stage1_label,stage2_label,source,type,split_key,split
0,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean,val
1,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean,train
2,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean,train
3,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean,train
4,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,Non_Organic,MendleyGCdataset,clean,Non_Organic_clean,val
...,...,...,...,...,...,...,...
67601,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean,train
67602,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean,train
67603,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean,train
67604,/Users/kalyanivenkateswaran/Documents/T3_Proje...,Non_Organic,White_Glass,garbage,clean,White_Glass_clean,val


In [32]:
import pandas as pd
import shutil
import os

BASE_DIR = Path.cwd()
STAGE1_DIR = BASE_DIR / "data" / "consolidated" / "Stage-1"
STAGE2_DIR = BASE_DIR / "data" / "consolidated" / "Stage-2"
META_DIR = BASE_DIR / "data" / "metadata"
file1=META_DIR/"consolidated_metatdata-stg1.csv"
target_dir = BASE_DIR / "data" / "consolidated" / "Stage-1" / "Test_batch"


os.makedirs(target_dir, exist_ok=True)

# read metadata
df = pd.read_csv(file1)

# filter test rows
test_df = df[df["split"] == "test"]   # change column name if needed

# copy files
for img_name in test_df["filename"]:   # change column name if needed
    
    src = os.path.join(source_dir, img_name)
    dst = os.path.join(target_dir, img_name)

    if os.path.exists(src):
        shutil.copy(src, dst)

print("Done copying test images")

NameError: name 'target_dir' is not defined

In [37]:
import os

BASE_DIR = Path.cwd()
root_dir = BASE_DIR / "data" / "raw" / "raw_reworked"

print(root_dir)
for foldername, subfolders, filenames in os.walk(root_dir):
    count = 0

    # count files in current folder + all subfolders
    for root, dirs, files in os.walk(foldername):
        count += len(files)

    print(f"{foldername} --> {count} files")

/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked --> 67635 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked/realwaste-main --> 4758 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked/realwaste-main/paper --> 500 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked/realwaste-main/misc_trash --> 513 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked/realwaste-main/textile --> 248 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked/realwaste-main/food_organics --> 411 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/raw/raw_reworked/realwaste-main/metal --> 789 files
/Users/kalyanivenkateswaran/

In [38]:
import os

BASE_DIR = Path.cwd()
root_dir = BASE_DIR / "Datasets - Final project for reorg"

print(root_dir)
for foldername, subfolders, filenames in os.walk(root_dir):
    count = 0

    # count files in current folder + all subfolders
    for root, dirs, files in os.walk(foldername):
        count += len(files)

    print(f"{foldername} --> {count} files")

/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg --> 205427 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg/realwaste-main --> 4759 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg/realwaste-main/paper --> 500 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg/realwaste-main/misc_trash --> 513 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg/realwaste-main/textile --> 248 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/Datasets - Final project for reorg/realwaste-main/food_organics --> 411 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classifi

In [39]:
import os

BASE_DIR = Path.cwd()
root_dir = BASE_DIR / "data" / "consolidated"

print(root_dir)
for foldername, subfolders, filenames in os.walk(root_dir):
    count = 0

    # count files in current folder + all subfolders
    for root, dirs, files in os.walk(foldername):
        count += len(files)

    print(f"{foldername} --> {count} files")

/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated --> 104480 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-1-Stage2-Deleted --> 3 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-1-duplicates --> 4294 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-1-duplicates/Organic --> 1613 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-1-duplicates/Non_Organic --> 2680 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2-Moved --> 69 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-1-Deleted --> 43 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classificat

In [40]:
import os

BASE_DIR = Path.cwd()
root_dir = BASE_DIR / "data" / "consolidated" / "Stage-2"

print(root_dir)
for foldername, subfolders, filenames in os.walk(root_dir):
    count = 0

    # count files in current folder + all subfolders
    for root, dirs, files in os.walk(foldername):
        count += len(files)

    print(f"{foldername} --> {count} files")

/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2 --> 33991 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2/Paper --> 1623 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2/Misc_Trash --> 1502 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2/Textile --> 5524 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2/Metal --> 2467 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2/E_waste --> 2277 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/data/consolidated/Stage-2/Medical_waste --> 1692 files
/Users/kalyanivenkateswaran/Documents/T3_Project/waste_classification/dat